# VCO tuning ranges

From the `vco_current_families()` runs in `results/vco/vco_families_*.csv` (a coarse code measured in several runs uses the newest) and the measured lookup table `LUT_CSV` (`vco_lut_measure()`). Per `vco_tune_coarse`, three bars, all measured:

- **Orange, band bottom:** f at VDD (Vctrl PMOS off) over `vco_current_min` 0..15, from the min family (max held at 0).
- **Blue, band top with min = 15:** f at Vctrl = 0 V over `vco_current_max` 0..15, from the max family.
- **Green, band top with min = 0:** f at Vctrl = 0 V over `vco_current_max` 0..15 with `vco_current_min` = 0, from the lookup table. It's only shown for coarse codes the table contains.

Ticks mark the individual codes; codes without a clock are left out.

In [156]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [157]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

VCO_DIR = Path.cwd().resolve().parents[2] / "results" / "vco"  # notebook lives in sw/tools/notebooks/
CSV_PATHS = None  # e.g. [VCO_DIR / "vco_families_20260928_185254.csv"]; None = all runs
LUT_CSV = VCO_DIR / "vco_lut_debug_20261001_154238.csv"  # measured lookup table; None = newest vco_lut_*.csv
LUT_CSV = None

if CSV_PATHS is None:
    CSV_PATHS = sorted(VCO_DIR.glob("vco_families_*.csv"))  # timestamped names: oldest first
df = pd.concat([pd.read_csv(p).assign(source_csv=Path(p).name) for p in CSV_PATHS], ignore_index=True)
newest = df.groupby("vco_tune_coarse")["source_csv"].max()  # coarse code in several runs: newest run
df = df[df["source_csv"] == df["vco_tune_coarse"].map(newest)]
coarse_codes = sorted(int(c) for c in df["vco_tune_coarse"].unique())
V_TOP, V_BOTTOM = float(df["vctrl_set"].min()), float(df["vctrl_set"].max())  # 0 V and VDD
print(f"{len(df)} points, coarse codes {coarse_codes}, Vctrl {V_TOP}..{V_BOTTOM} V, from {sorted(set(newest))}")

if LUT_CSV is None:
    found = sorted(VCO_DIR.glob("vco_lut_*.csv"), key=lambda p: p.stat().st_mtime)
    LUT_CSV = found[-1] if found else None
lut_df = pd.read_csv(LUT_CSV) if LUT_CSV else None
if lut_df is not None:
    print(f"lookup table {Path(LUT_CSV).name}: coarse {sorted(lut_df['vco_tune_coarse'].unique())}, "
          f"min {sorted(lut_df.loc[lut_df['swept'] == 'window', 'vco_current_min'].unique())}, "
          f"Vctrl {sorted(lut_df['vctrl'].unique())}")

TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.


8192 points, coarse codes [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], Vctrl 0.0..0.75 V, from ['vco_families_20260928_185254.csv']
lookup table vco_lut_S5_20261001_161638.csv: coarse [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)], min [np.int64(0), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)], Vctrl [np.float64(0.0), np.float64(0.2), np.float64(0.3), np.float64(0.35), np.float64(0.4), np.float64(0.45), np.float64(0.5), np.float64(0.55), np.float64(0.6), np.float64(0.65), np.float64(0.7), np.float64(0.75)]


In [158]:
import plotly.graph_objects as go


def f_at(coarse, field, vctrl):
    """{code: f_vco} of one family (field = the swept setting) at one Vctrl; NaN = no clock."""
    d = df[(df["vco_tune_coarse"] == coarse) & (df["swept"] == field) & np.isclose(df["vctrl_set"], vctrl)]
    return {int(k): float(f) for k, f in d.set_index(field)["f_vco"].items()}


def lut_top_min0(coarse):
    """{max code: f_vco} at Vctrl = V_TOP with vco_current_min = 0, from the measured lookup table ({} if absent)."""
    if lut_df is None:
        return {}
    d = lut_df[(lut_df["swept"] == "window") & (lut_df["vco_tune_coarse"] == coarse)
               & (lut_df["vco_current_min"] == 0) & np.isclose(lut_df["vctrl"], V_TOP)]
    return {int(k): float(f) for k, f in d.set_index("vco_current_max")["f_vco"].items()}


def with_clock(f):
    return {k: v for k, v in f.items() if not np.isnan(v) and v > 0}


INK, GRID = "#52514e", "#e4e3df"
# Frequency axes (values in MHz, log): labelled decades with the unit in the tick, minor grid in between.
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3, 1e4, 1e5],
                 ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz", "10 GHz", "100 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))


def add_range(fig, y, vals_by_code, color, name, code_name, unit, show_legend, height=0.26):
    """One horizontal bar from the smallest to the largest value, with a hoverable tick per code."""
    codes = sorted(vals_by_code)
    vals = np.array([vals_by_code[k] for k in codes])
    fig.add_bar(y=[y], x=[vals.max() - vals.min()], base=[vals.min()], orientation="h", width=height,
                marker_color=color, name=name, legendgroup=name, showlegend=show_legend,
                customdata=[[vals.min(), vals.max()]],
                hovertemplate=f"{name}<br>%{{customdata[0]:.1f}} - %{{customdata[1]:.1f}} {unit}<extra></extra>")
    fig.add_scatter(x=vals, y=[y] * len(vals), mode="markers", legendgroup=name, showlegend=False,
                    marker=dict(symbol="line-ns-open", size=12, color=INK, line_width=1), customdata=codes,
                    hovertemplate=f"{code_name} %{{customdata}}: %{{x:.1f}} {unit}<extra></extra>")


def style(fig, title, xtitle, ytitle="vco_tune_coarse", height=750, freq_x=False):
    fig.update_layout(title=title, template="plotly_white", height=height, barmode="overlay",
                      legend=dict(orientation="h", y=-0.12), hoverlabel=dict(namelength=-1))
    fig.update_xaxes(title=xtitle, gridcolor=GRID, **(FREQ_AXIS if freq_x else dict(type="log")))
    fig.update_yaxes(title=ytitle, tickvals=coarse_codes, autorange="reversed", gridcolor=GRID)


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) in results/vco/, then show the figure."""
    base = VCO_DIR / f"calibrate_vco_{name}"
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()


shown = set()  # legend entry once per bar type
fig = go.Figure()
for c in coarse_codes:
    rows = (
        (c - 0.3, "#eb6834", f_at(c, "vco_current_min", V_BOTTOM),
         f"bottom ({V_BOTTOM} V): vco_current_min 0..15, max = 0", "min"),
        (c, "#2a78d6", f_at(c, "vco_current_max", V_TOP),
         f"top ({V_TOP} V): vco_current_max 0..15, min = 15", "max"),
        (c + 0.3, "#1baf7a", lut_top_min0(c),
         f"top ({V_TOP} V): vco_current_max 0..15, min = 0 (lookup table)", "max"),
    )
    for y, color, f, name, code_name in rows:
        f = {k: v / 1e6 for k, v in with_clock(f).items()}
        if f:
            add_range(fig, y, f, color, name, f"coarse {c}, {code_name}", "MHz", show_legend=name not in shown)
            shown.add(name)
style(fig, "VCO tuning range per coarse code", "f_vco", freq_x=True)
save(fig, "tuning_ranges")

Chromium init'ed with kwargs {}
Found chromium path: /users/micas/wvandest/.local/share/choreographer/deps/chrome-linux64/chrome
Temp directory created: /tmp/tmposorm6cd.
Opening browser.
Temp directory created: /tmp/tmpdo3y_rd4.
Temporary directory at: /tmp/tmpdo3y_rd4
Conforming 1 to file:///tmp/tmposorm6cd/index.html
Getting tab from queue (has 1)
Got 5D1C
Reloading tab 5D1C before return.
Putting tab 5D1C back (queue size: 0).
Waiting for all cleanups to finish.
Exiting Kaleido.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
Cancelling tasks.
Exiting Kaleido/Choreo.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Cancelling tasks.
Exiting Kaleido/Choreo.


saved /users/micas/wvandest/projects/lagd-meas/results/vco/calibrate_vco_tuning_ranges.html and .png


In [159]:
# |Kvco| range per coarse code in the active Vctrl region: average slope over the window,
# |f(V_HI) - f(V_LO)| / (V_HI - V_LO), for every measured curve (max family and min family) with a clock at
# both edges. Ticks mark the individual curves (hover: which family and code).
V_LO, V_HI = 0.5, 0.7

fig = go.Figure()
for c in coarse_codes:
    kvco = {}
    for field, short in (("vco_current_max", "max"), ("vco_current_min", "min")):
        f_lo_edge, f_hi_edge = f_at(c, field, V_LO), f_at(c, field, V_HI)
        for code, a in f_lo_edge.items():
            b = f_hi_edge.get(code, np.nan)
            if not (np.isnan(a) or np.isnan(b)):
                kvco[f"{short} {code}"] = abs(b - a) / (V_HI - V_LO) / 1e6
    if kvco:
        add_range(fig, c, kvco, "#2a78d6", "|Kvco| over all measured settings", f"coarse {c},", "MHz/V",
                  show_legend=c == coarse_codes[0], height=0.5)
style(fig, f"|Kvco| range per coarse code, Vctrl {V_LO}-{V_HI} V (all measured current settings)", "|Kvco| [MHz/V]",
      height=650)
save(fig, "kvco_ranges")

TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Chromium init'ed with kwargs {}
Found chromium path: /users/micas/wvandest/.local/share/choreographer/deps/chrome-linux64/chrome
Temp directory created: /tmp/tmpq8oq4qv7.
Opening browser.
Temp directory created: /tmp/tmpiw5qee6t.
Temporary directory at: /tmp/tmpiw5qee6t
Conforming 1 to file:///tmp/tmpq8oq4qv7/index.html
Getting tab from queue (has 1)
Got 196E
Reloading tab 196E before return.
Putting tab 196E back (queue size: 0).
Waiting for all cleanups to finish.
Exiting Kaleido.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
Cancelling tasks.
Exiting Kaleido/Choreo.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Cancelling tasks.
Exiting Kaleido/Choreo.


saved /users/micas/wvandest/projects/lagd-meas/results/vco/calibrate_vco_kvco_ranges.html and .png


## configure_vco check: frequency + Kvco -> coarse, min, max

`configure_vco(f, kvco)` in `sw/tests/pll_test.py` picks a setting with `choose_vco` (in `sw/lib/vco_calibration.py`), using a measured lookup table of f(Vctrl) per (coarse, min, max) from `vco_lut_measure()`. The hard rules are applied first; among the settings that pass them, the one with Kvco closest to the target wins.
- **Rule 1:** the ring still oscillates at VDD.
- **Rule 2:** f is reached inside the Vctrl window, at least `margin` away from its edges.
- **Rule 3:** every table point inside the window has a clock.

The table is `LUT_CSV` from the setup cell at the top. There's no estimate from the family data: combining the min and max families doesn't give the frequency of a combined setting (the ring saturates when both branches conduct).

In [160]:
import logging
import sys

sys.path.insert(0, str(VCO_DIR.parents[1]))  # repo root
from sw.lib import vco_calibration

# Show choose_vco's report without switching on other libraries' logging (kaleido is very chatty).
_log = logging.getLogger(vco_calibration.__name__)
_log.setLevel(logging.INFO)
if not _log.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(message)s"))
    _log.addHandler(_handler)
    _log.propagate = False

TARGETS = [(32e6, 300e6), (256e6, 1e9), (512e6, 1e9), (1e9, 1e9), (3e9, 3e9), (10e9, 1e9)]  # (f_vco [Hz], |Kvco| [Hz/V])
WINDOW, MARGIN = vco_calibration.VCTRL_WINDOW, 0.02

if LUT_CSV is None:  # set in the setup cell
    raise FileNotFoundError(f"no vco_lut_*.csv in {VCO_DIR}: run vco_lut_measure() first")
lut = vco_calibration.read_lut(LUT_CSV)
print(f"table: {LUT_CSV}, {len(lut)} rows")

picks = []
for f_t, k_t in TARGETS:
    best, _ = vco_calibration.choose_vco(lut, f_t, k_t, window=WINDOW, margin=MARGIN)
    picks.append({"f_MHz": f_t / 1e6, "kvco_target_MHz_per_V": k_t / 1e6,
                  **({k: best[k] for k in vco_calibration.LUT_FIELDS} if best else {}),
                  "vctrl": best["vctrl"] if best else np.nan,
                  "kvco_MHz_per_V": best["kvco"] / 1e6 if best else np.nan,
                  "kvco_vs_target": best["kvco"] / k_t if best else np.nan,
                  "kvco_possible_min_MHz_per_V": best["kvco_min"] / 1e6 if best else np.nan,
                  "kvco_possible_max_MHz_per_V": best["kvco_max"] / 1e6 if best else np.nan,
                  "n_settings": best["n_options"] if best else 0,
                  "note": "" if best else "not reachable inside the window"})
picks = pd.DataFrame(picks)
picks

TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.


table: /users/micas/wvandest/projects/lagd-meas/results/vco/vco_lut_S5_20261001_161638.csv, 15570 rows
32.0 MHz: no setting reaches it inside Vctrl 0.00-0.75 V (margin 20 mV)
256.0 MHz: coarse 2, min 14, max 14 -> expected Vctrl 0.661 V, |Kvco| 987 MHz/V (target 1000 MHz/V, x0.99); possible |Kvco| at this frequency 349-7247 MHz/V (215 settings)
512.0 MHz: coarse 13, min 11, max 14 -> expected Vctrl 0.604 V, |Kvco| 991 MHz/V (target 1000 MHz/V, x0.99); possible |Kvco| at this frequency 84-12123 MHz/V (465 settings)
1000.0 MHz: coarse 7, min 11, max 14 -> expected Vctrl 0.485 V, |Kvco| 1031 MHz/V (target 1000 MHz/V, x1.03); possible |Kvco| at this frequency 16-21142 MHz/V (476 settings)
3000.0 MHz: coarse 0, min 9, max 14 -> expected Vctrl 0.481 V, |Kvco| 3145 MHz/V (target 3000 MHz/V, x1.05); possible |Kvco| at this frequency 1197-33302 MHz/V (111 settings)
10000.0 MHz: coarse 0, min 9, max 2 -> expected Vctrl 0.209 V, |Kvco| 987 MHz/V (target 1000 MHz/V, x0.99); possible |Kvco| at this

,f_MHz,kvco_target_MHz_per_V,vctrl,kvco_MHz_per_V,kvco_vs_target,kvco_possible_min_MHz_per_V,kvco_possible_max_MHz_per_V,n_settings,note,vco_tune_coarse,vco_current_min,vco_current_max
0,32.0,300.0,NaN,NaN,NaN,NaN,NaN,0,not reachable inside the window,NaN,NaN,NaN
1,256.0,1000.0,0.660653,987.456096,0.987456,348.56656,7247.06808,215,,2.0,14.0,14.0
2,512.0,1000.0,0.603608,990.758952,0.990759,84.23044,12123.29184,465,,13.0,11.0,14.0
3,1000.0,1000.0,0.485078,1030.981440,1.030981,16.05152,21142.03240,476,,7.0,11.0,14.0
4,3000.0,3000.0,0.481383,3144.690560,1.048230,1196.54400,33302.09280,111,,0.0,9.0,14.0
5,10000.0,1000.0,0.208667,987.088000,0.987088,885.19680,34809.61536,10,,0.0,9.0,2.0


In [161]:
# Which frequencies can be reached at all (same rules, any Kvco); gaps show up as separate bands.
freq_range = vco_calibration.vco_frequency_range(lut, window=WINDOW, margin=MARGIN)

TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.


coarse 0: 1429.0-12217.6 MHz
coarse 1: 698.8-6114.4 MHz
coarse 2: 210.5-3944.2 MHz
coarse 3: 192.0-2922.0 MHz
coarse 4: 179.1-2319.9 MHz
coarse 5: 167.1-1884.9 MHz
coarse 6: 76.3-1641.2 MHz
coarse 7: 72.8-1429.0 MHz
coarse 8: 69.5-1244.2 MHz
coarse 9: 66.4-1108.7 MHz
coarse 10: 63.4-1010.9 MHz
coarse 11: 60.6-921.8 MHz
coarse 12: 57.8-860.1 MHz
coarse 13: 56.5-802.6 MHz
coarse 14: 54.0-748.9 MHz
coarse 15: 52.7-698.8 MHz
reachable inside Vctrl 0.00-0.75 V (margin 20 mV): 52.7-12217.6 MHz


In [162]:
# The chosen settings: their measured table points (line) and the operating point (circle); window shaded.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]


def freq_label(f_mhz):
    return f"{f_mhz / 1e3:g} GHz" if f_mhz >= 1e3 else f"{f_mhz:g} MHz"


fig = go.Figure()
fig.add_vrect(x0=WINDOW[0], x1=WINDOW[1], fillcolor="#2a78d6", opacity=0.08, line_width=0)
y_data = []  # table points and operating points [MHz]: the vertical range follows these
for i, p in picks.dropna(subset=["vctrl"]).reset_index(drop=True).iterrows():
    key = tuple(int(p[k]) for k in vco_calibration.LUT_FIELDS)
    color = PALETTE[i % len(PALETTE)]
    name = f"{p['f_MHz']:.0f} MHz: c{key[0]} min {key[1]} max {key[2]}"
    pts = sorted((r["vctrl"], r["f_vco"]) for r in lut if tuple(r[k] for k in vco_calibration.LUT_FIELDS) == key)
    v, f = np.array(pts).T
    fig.add_scatter(x=v, y=f / 1e6, mode="lines+markers", line=dict(color=color, width=2), marker_size=6,
                    name=name, legendgroup=name,
                    hovertemplate=f"{name}<br>Vctrl %{{x:.3f}} V: %{{y:.1f}} MHz<extra></extra>")
    fig.add_scatter(x=[p["vctrl"]], y=[p["f_MHz"]], mode="markers", legendgroup=name, showlegend=False,
                    marker=dict(size=14, color="rgba(0,0,0,0)", line=dict(color=color, width=2)),
                    hovertemplate=(f"{name}<br>operating point: Vctrl %{{x:.3f}} V<br>"
                                   f"|Kvco| {p['kvco_MHz_per_V']:.0f} MHz/V (target {p['kvco_target_MHz_per_V']:.0f}, "
                                   f"possible {p['kvco_possible_min_MHz_per_V']:.0f}-"
                                   f"{p['kvco_possible_max_MHz_per_V']:.0f})<extra></extra>"))
    y_data += [x for x in f / 1e6 if x > 0] + [p["f_MHz"]]

# Vertical range: the table and operating points with some room; 1-2-5 ticks so a narrow range is labelled too.
y_axis = dict(FREQ_AXIS)
if y_data:
    lo, hi = min(y_data) / 1.3, max(y_data) * 1.3
    ticks = [m * 10 ** e for e in range(0, 6) for m in (1, 2, 5) if lo <= m * 10 ** e <= hi]
    y_axis.update(range=[np.log10(lo), np.log10(hi)], tickvals=ticks, ticktext=[freq_label(t) for t in ticks])
fig.update_layout(title="configure_vco picks: f_vco vs Vctrl (circle = operating point, shaded = window)",
                  template="plotly_white", height=600, legend=dict(orientation="h", y=-0.15),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="Vctrl [V]", range=[WINDOW[0] - 0.02, max(r["vctrl"] for r in lut) + 0.02], gridcolor=GRID)
fig.update_yaxes(title="f_vco", gridcolor=GRID, **y_axis)
save(fig, "configure_picks")

Chromium init'ed with kwargs {}
Found chromium path: /users/micas/wvandest/.local/share/choreographer/deps/chrome-linux64/chrome
Temp directory created: /tmp/tmp_8q4l66o.
Opening browser.
Temp directory created: /tmp/tmpektt40r7.
Temporary directory at: /tmp/tmpektt40r7
Conforming 1 to file:///tmp/tmp_8q4l66o/index.html
Getting tab from queue (has 1)
Got 524E
Reloading tab 524E before return.
Putting tab 524E back (queue size: 0).
Waiting for all cleanups to finish.
Exiting Kaleido.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
Cancelling tasks.
Exiting Kaleido/Choreo.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Cancelling tasks.
Exiting Kaleido/Choreo.


saved /users/micas/wvandest/projects/lagd-meas/results/vco/calibrate_vco_configure_picks.html and .png


## Drift during the lookup-table run

`vco_lut_measure()` measures fixed reference settings (`LUT_DRIFT_REFS` in `pll_test.py`) at the start and after every coarse code. By default that's coarse 8, min 8, max 8 at two Vctrl values:
- **0 V:** the curve is flat there, so only the VCO itself (temperature, supply) can move it;
- **0.65 V:** the curve is steep there, so drift of the current source or Vctrl shows up as well.

The plot shows how far each reference moved from its first reading. Error bars are the spread of the scope readings. If only the 0.65 V line moves, the cause is on the Vctrl/current-source side; if both move by about the same amount, it's the VCO. Tables measured before this was added have no drift rows.

In [163]:
drift_df = lut_df[lut_df["swept"] == "drift"].copy() if lut_df is not None and "time" in lut_df else None
if drift_df is None or drift_df.empty:
    print(f"no drift reference in {Path(LUT_CSV).name if LUT_CSV else 'the lookup table'} (measured before it was added)")
else:
    drift_df["time"] = pd.to_datetime(drift_df["time"])
    # Each drift round measures every reference once: label the rounds by the coarse code measured just before.
    order = list(dict.fromkeys(lut_df.loc[lut_df["swept"] != "drift", "vco_tune_coarse"]))
    labels = ["start"] + [f"after coarse {c}" for c in order]
    ref_keys = ["vco_tune_coarse", "vco_current_min", "vco_current_max", "vctrl"]
    DRIFT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"]

    fig = go.Figure()
    fig.add_hline(y=0, line=dict(color=INK, width=1))
    for i, (key, d) in enumerate(drift_df.groupby(ref_keys, sort=False)):
        d = d.sort_values("time")
        f0 = d["f_vco"].iloc[0]
        dev = (d["f_vco"] / f0 - 1) * 100
        name = f"coarse {key[0]}, min {key[1]}, max {key[2]}, Vctrl {key[3]:.2f} V (start {f0 / 1e6:.2f} MHz)"
        fig.add_scatter(x=d["time"], y=dev, mode="lines+markers", name=name,
                        line=dict(color=DRIFT_COLORS[i % len(DRIFT_COLORS)], width=2), marker_size=7,
                        error_y=dict(type="data", array=d["f_std"] / d["f_meas"] * 100, thickness=1.5),
                        customdata=list(zip(labels[:len(d)], d["f_vco"] / 1e6)),
                        hovertemplate=(f"Vctrl {key[3]:.2f} V, %{{customdata[0]}}<br>%{{x|%H:%M}}<br>"
                                       "%{customdata[1]:.2f} MHz (%{y:+.2f} %)<extra></extra>"))
        print(f"{name}: largest change {dev.abs().max():.2f} % over {d['time'].iloc[-1] - d['time'].iloc[0]}")
    fig.update_layout(title="Drift references during the lookup-table run", template="plotly_white", height=480,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="time", gridcolor=GRID)
    fig.update_yaxes(title="f_vco change from the first reading [%]", gridcolor=GRID, zeroline=False)
    save(fig, "lut_drift")

coarse 8, min 8, max 8, Vctrl 0.00 V (start 1225.44 MHz): largest change 0.08 % over 0 days 11:45:59
coarse 8, min 8, max 8, Vctrl 0.65 V (start 807.52 MHz): largest change 0.25 % over 0 days 11:45:59


TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Chromium init'ed with kwargs {}
Found chromium path: /users/micas/wvandest/.local/share/choreographer/deps/chrome-linux64/chrome
Temp directory created: /tmp/tmp34q0w606.
Opening browser.
Temp directory created: /tmp/tmp761oh88z.
Temporary directory at: /tmp/tmp761oh88z
Conforming 1 to file:///tmp/tmp34q0w606/index.html
Getting tab from queue (has 1)
Got 6765
Reloading tab 6765 before return.
Putting tab 6765 back (queue size: 0).
Waiting for all cleanups to finish.
Exiting Kaleido.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Closing browser.
Cancelling tasks.
Exiting Kaleido/Choreo.
TemporaryDirectory.cleanup() worked.
shutil.rmtree worked.
Cancelling tasks.
Exiting Kaleido/Choreo.


saved /users/micas/wvandest/projects/lagd-meas/results/vco/calibrate_vco_lut_drift.html and .png
